# Phase A — Edge Detection

Objectif : déterminer si Polymarket BTC Up/Down 5min est efficient ou si un edge statistique existe.

**Questions :**
1. Le prix Polymarket YES prédit-il mieux que 50/50 ?
2. Peut-on battre le marché avec un modèle simple (momentum, orderbook) ?
3. Quelle est la calibration du marché (Brier score) ?

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.features import build_features, FEATURE_COLS, get_xy
from src.models.lgbm import train

plt.style.use('dark_background')

## 1. Charger les ticks collectés

In [ ]:
conn = sqlite3.connect('../data/bot.db')
df = pd.read_sql('SELECT * FROM ticks ORDER BY ts', conn)
conn.close()

df['dt'] = pd.to_datetime(df['ts'], unit='ms')
print(f'Ticks: {len(df):,}')
print(f'Période: {df["dt"].min()} → {df["dt"].max()}')
df.head()

## 2. Features + label

In [ ]:
feats = build_features(df)
X, y = get_xy(feats)
print(f'X shape: {X.shape} | y base rate: {y.mean():.3f}')

## 3. Baseline : le marché Polymarket est-il calibré ?

On vérifie Brier score — si < 0.25 le marché est mieux que 50/50, s'il est à 0.25 c'est du pur hasard.

In [ ]:
from sklearn.metrics import brier_score_loss, log_loss

clean = feats.dropna(subset=['future_price']).copy()
market_probs = clean['poly_yes'].clip(0.01, 0.99)
outcomes = clean['label']

print(f'Brier marché : {brier_score_loss(outcomes, market_probs):.4f} (0.25 = random, < 0.25 = info)')
print(f'LogLoss marché : {log_loss(outcomes, market_probs):.4f}')
print(f'Brier 50/50 naive : {brier_score_loss(outcomes, np.full_like(outcomes, 0.5, dtype=float)):.4f}')

## 4. Entraînement LightGBM

In [ ]:
result = train(feats)
print('Métriques:', result.metrics)
result.feature_importance

## 5. Courbe de calibration

In [ ]:
from sklearn.calibration import calibration_curve

probs_model = result.model.predict_proba(X.iloc[int(len(X)*0.8):])[:, 1]
y_test = y.iloc[int(len(y)*0.8):]

frac_pos, mean_pred = calibration_curve(y_test, probs_model, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Parfait')
ax.plot(mean_pred, frac_pos, 'o-', label='Modèle LGBM')
ax.set_xlabel('Prédit')
ax.set_ylabel('Observé')
ax.set_title('Calibration du modèle')
ax.legend()
plt.show()

## 6. Simulation trading (edge détecté)

In [ ]:
from src.simulator.paper import PaperSimulator

sim = PaperSimulator()
test_feats = feats.iloc[int(len(feats)*0.8):].copy()
test_feats['model_prob'] = result.model.predict_proba(test_feats[FEATURE_COLS].fillna(0))[:, 1]

for _, row in test_feats.dropna(subset=['future_price']).iterrows():
    side = 'YES' if row['model_prob'] > row['poly_yes'] else 'NO'
    price = row['poly_yes'] if side == 'YES' else (1 - row['poly_yes'])
    t = sim.place_trade(int(row['ts']), 'BTC-5m', side, price, float(row['model_prob']))
    if t:
        sim.resolve(t, btc_went_up=bool(row['label']))

print(sim.summary())